In [1]:
import gc
import os
import sys

import numpy as np
import optuna
import pandas as pd
import torch

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
device.type

'cpu'

In [3]:
%load_ext autoreload
%autoreload 2

from drGAT import drGAT

In [4]:
train_data = pd.read_csv("../CTRP_data/train.csv")
val_data = pd.read_csv("../CTRP_data/val.csv")
test_data = pd.read_csv("../CTRP_data/test.csv")

In [5]:
train_data.head()

,DRUG_NAME,CELL_LINE_NAME
0,BRD-K27986637,RERFLCAI
1,BRD-K09344309,NCIH1666
2,quizartinib,OVTOKO
3,BRD-K99584050,SNU878
4,ML210,YD15


In [6]:
idxs = np.load("../CTRP_data/idxs.npy", allow_pickle=True)
converter = {idxs[1, i]: int(idxs[0, i]) for i in range(idxs.shape[1])}

In [7]:
edge_index = np.load("../CTRP_data/edge_idxs.npy")
edge_attr = np.load("../CTRP_data/edge_attr.npy")

In [8]:
def get_idx(X):
    X["Drug"] = [converter[(i)] for i in X["DRUG_NAME"]]
    X["Cell"] = [converter[(i)] for i in X["CELL_LINE_NAME"]]
    return X

In [9]:
train_data = get_idx(train_data)
val_data = get_idx(val_data)
test_data = get_idx(test_data)

In [10]:
edge_index = torch.tensor(edge_index).int()
edge_index = edge_index.type(torch.int64)

In [11]:
edge_attr = torch.tensor(edge_attr).float()

In [12]:
train_drug = train_data["Drug"].values
train_cell = train_data["Cell"].values
val_drug = val_data["Drug"].values
val_cell = val_data["Cell"].values

In [13]:
train_labels = np.load("../CTRP_data/train_labels.npy")
val_labels = np.load("../CTRP_data/val_labels.npy")

train_labels = torch.tensor(train_labels).float()
val_labels = torch.tensor(val_labels).float()

## Get feature matrix

In [14]:
drug = pd.read_csv("../CTRP_data/drug_sim.csv", index_col=0)
cell = pd.read_csv("../CTRP_data/cell_sim.csv", index_col=0)
gene = pd.read_csv("../CTRP_data/gene_sim.csv", index_col=0)

In [15]:
drug = torch.tensor(drug.values).float()
cell = torch.tensor(cell.values).float()
gene = torch.tensor(gene.values).float()

# Create the dataset

In [16]:
data = [
    drug,
    cell,
    gene,
    edge_index,
    edge_attr,
    train_drug,
    train_cell,
    val_drug,
    val_cell,
    train_labels,
    val_labels,
]

In [17]:
params = {
    "dropout1": 0.1,
    "dropout2": 0.1,
    "n_drug": drug.shape[0],
    "n_cell": cell.shape[0],
    "n_gene": gene.shape[0],
    "hidden1": 256,
    "hidden2": 32,
    "hidden3": 128,
    "epochs": 3,
    "lr": 0.001,
    "heads": 2,
}

In [57]:
def objective(trial):
    params = {
        "n_drug": drug.shape[0],
        "n_cell": cell.shape[0],
        "n_gene": gene.shape[0],
        "dropout1": trial.suggest_categorical("dropout1", [0.1, 0.2, 0.3, 0.4, 0.5]),
        "dropout2": trial.suggest_categorical("dropout2", [0.1, 0.2, 0.3, 0.4, 0.5]),
        "hidden1": trial.suggest_categorical(
            "hidden1",
            [
                10
                # 256,
                # 512,
                # 1028
            ],
        ),
        "hidden2": trial.suggest_categorical(
            "hidden2",
            [
                10
                # 128,
                # 256,
                # 512,
            ],
        ),
        "hidden3": trial.suggest_categorical(
            "hidden3",
            [
                10
                # 64,
                # 128,
                # 256,
            ],
        ),
        "epochs": trial.suggest_int("epochs", 2, 3, step=1),
        # "epochs": trial.suggest_int("epochs", 10, 200, step=50),
        "heads": trial.suggest_categorical("heads", [1, 2, 3]),
        "activation": trial.suggest_categorical(
            "activation", ["relu", "gelu", "swish"]
        ),
        "optimizer": trial.suggest_categorical("optimizer", ["Adam", "AdamW", "SGD"]),
        "lr": trial.suggest_float("lr", 1e-5, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "scheduler": trial.suggest_categorical(
            "scheduler", [None, "Cosine", "Step", "Plateau"]
        ),
        "gnn_layer": trial.suggest_categorical(
            "gnn_layer", ["GAT", "GATv2", "Transformer"]
        ),
    }

    # スケジューラ関連パラメータの条件付き追加
    if params["scheduler"] == "Cosine":
        params["T_max"] = trial.suggest_int("T_max", 20, 50)
    elif params["scheduler"] == "Step":
        params["scheduler_gamma"] = trial.suggest_float("gamma_step", 0.1, 0.95)
        params["step_size"] = trial.suggest_int("step_size", 10, 30)
    elif params["scheduler"] == "Plateau":
        params["patience"] = trial.suggest_int("patience_plateau", 3, 10)
        params["threshold"] = trial.suggest_float(
            "thresh_plateau", 1e-4, 1e-2, log=True
        )

    if params["hidden1"] < params["hidden2"] or params["hidden2"] < params["hidden3"]:
        raise optuna.TrialPruned("Invalid layer size configuration")

    if params["optimizer"] in ["Adam", "AdamW"]:
        params["amsgrad"] = trial.suggest_categorical("amsgrad", [True, False])

    if params["optimizer"] == "SGD":
        params["momentum"] = trial.suggest_float("momentum", 0.8, 0.95)
        params["nesterov"] = trial.suggest_categorical("nesterov", [True, False])

    # 隠れ層サイズとバッチサイズの関係を制約
    if (params["hidden1"] > 512) and (params["hidden2"] > 256):
        raise optuna.TrialPruned("Memory intensive configuration")

    try:
        _, _, _, best_metrics, early_stopping_epoch = drGAT.train(
            data, params=params, device=device, verbose=False
        )

        early_stop_threshold = trial.suggest_float("early_stop_threshold", 0.3, 0.7)
        if (
            early_stopping_epoch is not None
            and early_stopping_epoch < params["epochs"] * early_stop_threshold
        ):
            raise optuna.TrialPruned("Early stopping occurred too early")

        trial.set_user_attr("early_stopping_epoch", early_stopping_epoch)
        return best_metrics

    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            print("CUDA out of memory")
            trial.set_user_attr("status", "CUDA OOM")

            torch.cuda.empty_cache()
            gc.collect()

            return [float("-inf")] * 4
        else:
            raise e

In [ ]:
name = "CTRP"
study = optuna.create_study(
    directions=["maximize"] * 4,
    sampler=optuna.samplers.TPESampler(),
    pruner=optuna.pruners.HyperbandPruner(),
    storage="sqlite:///{}.sqlite3".format(name),
    study_name=name,
    load_if_exists=True,
)
study.optimize(objective, n_trials=100)

[I 2025-03-20 11:58:36,136] A new study created in RDB with name: CTRP
/Users/inouey2/code/drGAT/.venv/lib/python3.12/site-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


Using:  cpu


100%|███████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:03<00:00,  1.52s/it]
[I 2025-03-20 11:58:39,250] Trial 0 finished with values: [0.5, 0.639775381058177, 0.6687076336113106, 0.6666666666666666] and parameters: {'dropout1': 0.3, 'dropout2': 0.4, 'hidden1': 10, 'hidden2': 10, 'hidden3': 10, 'epochs': 2, 'heads': 2, 'activation': 'swish', 'optimizer': 'Adam', 'lr': 1.408533519464261e-05, 'weight_decay': 0.0019805804490557613, 'scheduler': 'Step', 'gnn_layer': 'GAT', 'gamma_step': 0.6974481493588492, 'step_size': 28, 'amsgrad': True, 'early_stop_threshold': 0.6359872695666402}.
/Users/inouey2/code/drGAT/.venv/lib/python3.12/site-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


Best model found at epoch 2
Using:  cpu


100%|███████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.16it/s]
[I 2025-03-20 11:58:41,035] Trial 1 finished with values: [0.6256968171192232, 0.6736018621960541, 0.687770635020442, 0.6773118362917603] and parameters: {'dropout1': 0.1, 'dropout2': 0.1, 'hidden1': 10, 'hidden2': 10, 'hidden3': 10, 'epochs': 2, 'heads': 1, 'activation': 'swish', 'optimizer': 'SGD', 'lr': 0.009511714950113938, 'weight_decay': 0.0003251934404160963, 'scheduler': 'Cosine', 'gnn_layer': 'Transformer', 'T_max': 32, 'momentum': 0.8083760317956223, 'nesterov': True, 'early_stop_threshold': 0.618915518626154}.
/Users/inouey2/code/drGAT/.venv/lib/python3.12/site-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


...

100%|███████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:03<00:00,  1.52s/it]
[I 2025-03-20 12:00:23,603] Trial 35 finished with values: [0.5, 0.5823046172087768, 0.6045152678984558, 0.6666666666666666] and parameters: {'dropout1': 0.5, 'dropout2': 0.3, 'hidden1': 10, 'hidden2': 10, 'hidden3': 10, 'epochs': 2, 'heads': 2, 'activation': 'gelu', 'optimizer': 'Adam', 'lr': 0.005991683286217483, 'weight_decay': 0.00013346848453086227, 'scheduler': 'Step', 'gnn_layer': 'GAT', 'gamma_step': 0.328266231797309, 'step_size': 11, 'amsgrad': True, 'early_stop_threshold': 0.3970327060863907}.
/Users/inouey2/code/drGAT/.venv/lib/python3.12/site-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


Best model found at epoch 2
Using:  cpu


100%|███████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:03<00:00,  1.54s/it]
[I 2025-03-20 12:00:26,760] Trial 36 finished with values: [0.5, 0.3663641285247623, 0.23131597533248563, 0.0] and parameters: {'dropout1': 0.1, 'dropout2': 0.3, 'hidden1': 10, 'hidden2': 10, 'hidden3': 10, 'epochs': 2, 'heads': 2, 'activation': 'gelu', 'optimizer': 'Adam', 'lr': 0.003068477953540414, 'weight_decay': 3.564960560581439e-05, 'scheduler': None, 'gnn_layer': 'GAT', 'amsgrad': True, 'early_stop_threshold': 0.610291044981323}.
/Users/inouey2/code/drGAT/.venv/lib/python3.12/site-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


Best model found at epoch 2
Using:  cpu


  0%|                                                                                                   | 0/2 [00:00<?, ?it/s]

## Eval

In [30]:
# test_drug = test_data.values[:, 0]
# test_cell = test_data.values[:, 1]

# test_labels = np.load("data/test_labels.npy")
# test_labels = torch.tensor(test_labels).float()
# test = [drug, cell, gene, edge_index, test_drug, test_cell, test_labels]

In [63]:
# prob, res, test_attention = drGAT.eval(model, test)

['maximizemaximizemaximizemaximize']